# Why Mean-Variance Optimization Breaks

Module 04 derived the tangency portfolio $w \propto \Sigma^{-1}(\mu - r_f)$ assuming $\mu$ and $\Sigma$ are known. Module 06 showed that $\mu$ is barely known at
all. Put the two together and the optimizer becomes an "error maximizer" (Michaud 1989): it puts the most weight where the
estimated returns are too high and the estimated risks too low. This notebook shows where the amplification comes from (the
inverse covariance matrix), how optimistic the estimated frontier is, why the naive 1/N portfolio is hard to beat, and
what long-only constraints do. It sets up the remedies of 08/02–08/04.

## 0. Math link

- Estimation error of means and covariances from module 06, in particular
  [06/01](../06_estimation_error/01_how_noisy_is_a_mean_return.ipynb) and
  [06/04](../06_estimation_error/04_means_are_hard_covariances_are_easier.ipynb).
- The tangency portfolio and the Sharpe ratio from [04/04](../04_portfolio_mathematics/04_efficient_frontier_tangency_sharpe.ipynb).
- [Spectral decomposition](https://github.com/dnlwrthstr/LearningLinearAlgebra/blob/main/hyperskill/05_matrix_decomposition/055_spectral_decomposition.ipynb):
  $\Sigma^{-1} = V\Lambda^{-1}V^\top$ shows which directions the inverse amplifies.
- [Expected value of a quadratic form](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/03_probability_distribution_with_multiple_variables/08_covariance_matrix.ipynb):
  $E[e^\top Ae] = \operatorname{tr}(A\operatorname{Cov}(e))$ for zero-mean $e$, used for the in-sample bias.

## 1. Introduction

An analyst feeds ten years of historical returns for eight asset classes into an optimizer. The result: 150% in one bond
index, −120% in a nearly identical one, and an in-sample Sharpe ratio twice that of any asset class. Next year the
portfolio disappoints. Nothing is wrong with the optimizer. It did exactly what it was told, with inputs that were mostly
noise.

After this notebook you can:

- show how $\Sigma^{-1}$ amplifies errors in expected returns, using its eigenvalues;
- derive the upward bias of the in-sample maximum Sharpe ratio, about $N/T$ in squared Sharpe units;
- compute the out-of-sample Sharpe ratio of a plug-in portfolio and compare it with 1/N;
- see how long-only constraints turn wild weights into corner solutions;
- explain why many practitioners use 1/N, risk-based portfolios or shrinkage instead of raw optimization.

## 2. Theory

### The plug-in portfolio

With excess returns $m = \mu - r_f$, the tangency direction is $w^* \propto \Sigma^{-1}m$ and the maximum Sharpe ratio is $\mathrm{SR}^* = \sqrt{m^\top\Sigma^{-1}m}$. The
plug-in portfolio replaces $m$ by the estimate $\hat m$ (and often $\Sigma$ by $\hat\Sigma$).

### Amplification through $\Sigma^{-1}$

With $\Sigma = \sum_k \lambda_k v_kv_k^\top$:

$$
\Sigma^{-1}\hat m = \sum_k \frac{v_k^\top\hat m}{\lambda_k}\,v_k
$$

The weight along eigenvector $v_k$ is the expected return along that direction divided by its variance. Small eigenvalues
belong to **spread** portfolios of similar assets (two bond indices, two equity indices), whose true expected return is
small. An estimation error $e = \hat m - m$ projected on such a direction is divided by a small $\lambda_k$ and produces a large
long-short position. The condition number $\lambda_1/\lambda_N$ measures how unequal the amplification is.

### The in-sample Sharpe ratio is too high

Suppose $\Sigma$ is known and $\hat m = m + e$ with $e \sim N(0, \Sigma/T)$ ($T$ years of data, annual units). The estimated maximum squared Sharpe ratio is

$$
\widehat{\mathrm{SR}}^2 = \hat m^\top\Sigma^{-1}\hat m = m^\top\Sigma^{-1}m + 2m^\top\Sigma^{-1}e + e^\top\Sigma^{-1}e
$$

The middle term has mean zero, and $E[e^\top\Sigma^{-1}e] = \operatorname{tr}(\Sigma^{-1}\Sigma/T) = N/T$. Hence

$$
E\big[\widehat{\mathrm{SR}}^2\big] = \mathrm{SR}^{*2} + \frac{N}{T}
$$

With eight assets and ten years, the in-sample squared Sharpe ratio is inflated by 0.8, whatever the true one is.

### The out-of-sample Sharpe ratio is too low

The plug-in direction $\hat w = \Sigma^{-1}\hat m$ has true Sharpe ratio

$$
\mathrm{SR}(\hat w) = \frac{m^\top\Sigma^{-1}\hat m}{\sqrt{\hat m^\top\Sigma^{-1}\hat m}} \le \mathrm{SR}^*
$$

(Cauchy–Schwarz in the $\Sigma^{-1}$ inner product), with equality only if $\hat m \propto m$. The noisier $\hat m$, the further below $\mathrm{SR}^*$; it can be
negative.

### The 1/N benchmark

Equal weights use no estimates at all. DeMiguel, Garlappi and Uppal (2009) found that none of 14 optimized strategies
consistently beat 1/N out of sample in their data sets, and estimated that a plug-in tangency portfolio would need
centuries of data to win reliably with 25 assets.

### Constraints

Long-only constraints ($w \ge 0$) cap the damage: the optimizer can no longer build long-short spreads. But the solution
tends to sit in a **corner**: a few assets with the highest estimated return per unit of risk get all the weight, and which
ones changes with each re-estimation. Jagannathan and Ma (2003) showed that a constraint acts like a form of shrinkage of
the covariance matrix.

## 3. Worked Examples

### Example 1: two nearly identical assets

Two assets with volatility 10% and true excess return 5% each. Uncorrelated: optimum 50/50. Estimated excess returns 6% and 4%:
$w \propto (6, 4)$, i.e. 60/40.

With correlation $\rho = 0.9$: $\Sigma^{-1} \propto \begin{pmatrix}1 & -\rho\\ -\rho & 1\end{pmatrix}$, so $w \propto (6 - 0.9\cdot 4,\ 4 - 0.9\cdot 6) = (2.4, -1.4)$, normalized to sum 1:
**240% and −140%**. A 2-point estimation error (well within one standard error from 06/01) turns 50/50 into a leveraged spread.

### Example 2: the eigenvalue view

Same assets, $\rho = 0.9$: eigenvalues $0.01(1 \pm 0.9) = 0.019$ (sum direction) and $0.001$ (spread direction). The estimated spread return
$(6 - 4)/\sqrt2 = 1.41\%$ is divided by 0.001, the sum return $10/\sqrt2 = 7.07\%$ by 0.019. The spread position is $1.41/0.001 = 1{,}414$ against
$7.07/0.019 = 372$ for the sum: the noise dominates.

### Example 3: in-sample optimism

$N = 8$ assets, $T = 10$ years, true $\mathrm{SR}^* = 0.55$: $E[\widehat{\mathrm{SR}}^2] = 0.30 + 0.8 = 1.10$, so the in-sample Sharpe ratio is about $\sqrt{1.10} = 1.05$,
nearly double the truth.

### Example 4: when could the plug-in beat 1/N?

If 1/N has Sharpe ratio 0.48 and the true maximum is 0.55, the plug-in must recover most of a gap of only 0.07. With $N/T$ of
noise in the squared Sharpe ratio, that needs many decades: the simulation below finds about 80 years for eight assets, even with
the covariance matrix known exactly.

## 4. Finance Application

### Where it matters

- **Strategic asset allocation.** Optimizing on historical means produces allocations no committee would sign. Practice
  uses equilibrium returns (08/04), constraints, and judgement.
- **Backtested strategies.** In-sample Sharpe ratios of optimized portfolios are inflated by about $N/T$; report
  out-of-sample results only.
- **Similar assets.** The most extreme positions are between near-duplicates (two bond indices, two equity regions).
  Grouping or removing near-duplicates reduces the amplification.
- **Turnover.** Re-estimating each year moves the weights a lot even when nothing true has changed; transaction costs add to
  the loss.

### The usual misreadings

- "The optimizer found an inefficiency." It most likely found an estimation error in a low-variance spread.
- "Adding more assets can only help the optimizer." Each asset adds $1/T$ to the in-sample squared Sharpe ratio and more noise to
  the weights; out of sample, more assets can hurt.
- "Long-only constraints fix the problem." They limit leverage but leave concentrated, unstable corner portfolios.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

rng = np.random.default_rng(42)

assets = ["Swiss eq", "global eq", "EM eq", "Swiss small caps", "CHF bonds", "global bonds (hedged)", "CHF corporates", "Swiss real estate"]
N = len(assets)
vols = np.array([0.16, 0.17, 0.22, 0.19, 0.05, 0.045, 0.06, 0.12])
mu = np.array([0.065, 0.07, 0.08, 0.075, 0.02, 0.022, 0.028, 0.05])
rf = 0.005
corr = np.eye(N)
pairs = {(0, 1): 0.80, (0, 2): 0.65, (0, 3): 0.85, (1, 2): 0.75, (1, 3): 0.70, (2, 3): 0.60,
         (4, 5): 0.75, (4, 6): 0.85, (5, 6): 0.70}
for (i, j), v in pairs.items():
    corr[i, j] = corr[j, i] = v
for i in range(4):
    for j in range(4, 7):
        corr[i, j] = corr[j, i] = 0.25 if j == 6 else 0.10
    corr[i, 7] = corr[7, i] = 0.5
for j in range(4, 7):
    corr[j, 7] = corr[7, j] = 0.3
Sigma = np.outer(vols, vols) * corr
m = mu - rf
print("min eigenvalue of the correlation matrix:", np.linalg.eigvalsh(corr).min().round(3))

### The true optimum and 1/N

In [ ]:
def tangency(m_in, S_in):
    x = np.linalg.solve(S_in, m_in)
    return x / x.sum()

def sharpe(w, m_true=m, S_true=Sigma):
    return (w @ m_true) / np.sqrt(w @ S_true @ w)

w_true = tangency(m, Sigma)
sr_max = np.sqrt(m @ np.linalg.solve(Sigma, m))
w_eq = np.full(N, 1 / N)

lam, V = np.linalg.eigh(Sigma)
print(f"max Sharpe {sr_max:.3f}, 1/N Sharpe {sharpe(w_eq):.3f}")
print(f"eigenvalues of Sigma: {lam.round(5)}")
print(f"condition number {lam[-1] / lam[0]:.0f}")
print("smallest-eigenvalue direction:", dict(zip(assets, V[:, 0].round(2))))
pd.Series(w_true, index=assets).map(lambda v: f"{v:.1%}")

The true tangency portfolio already shorts CHF bonds (−32%) against CHF corporates and global bonds: with correlations of 0.7–0.85
among the bond indices, small differences in their expected returns justify spreads. The smallest eigenvalue, 0.00036, belongs
to exactly such a spread (long corporates, short CHF bonds); the condition number is 316.

### Sensitivity: one percentage point on one asset

Raise the expected return of global bonds by 1 point (true $\Sigma$) and recompute the tangency weights.

In [ ]:
m_bump = m.copy()
m_bump[5] += 0.01
w_bump = tangency(m_bump, Sigma)
pd.DataFrame({"true weights": w_true, "global bonds +1%": w_bump, "change": w_bump - w_true}, index=assets) \
    .style.format("{:.1%}")

One extra percentage point of expected return on global bonds, a fraction of its ten-year standard error, doubles its weight
from 67% to 137%, doubles the CHF bond short from −32% to −63%, and cuts CHF corporates from 36% to 7%. The equity weights barely move:
the amplification happens among the near-duplicates.

### Plug-in portfolios from ten years of data

1,000 histories of 10 years of monthly returns. For each: the plug-in tangency with estimated $\mu$ and true $\Sigma$, and with both
estimated. Evaluate in-sample and true (out-of-sample) Sharpe ratios.

In [ ]:
n_hist, T = 1_000, 10
L = np.linalg.cholesky(Sigma / 12)
rows, W_plug = [], []
for h in range(n_hist):
    R = mu / 12 + rng.standard_normal((T * 12, N)) @ L.T
    m_hat = R.mean(axis=0) * 12 - rf
    S_hat = np.cov(R, rowvar=False) * 12
    w_mu = np.linalg.solve(Sigma, m_hat)
    w_both = np.linalg.solve(S_hat, m_hat)
    W_plug.append(w_both / w_both.sum())
    rows.append({
        "in-sample SR, mu estimated": np.sqrt(m_hat @ w_mu),
        "true SR, mu estimated": sharpe(w_mu),
        "in-sample SR, both estimated": np.sqrt(m_hat @ w_both),
        "true SR, both estimated": sharpe(w_both),
    })
res = pd.DataFrame(rows)
W_plug = np.array(W_plug)

print(f"theory: E[in-sample SR^2] = {sr_max**2:.3f} + {N}/{T} = {sr_max**2 + N / T:.3f}")
print(f"simulated mean in-sample SR^2 (mu estimated): {(res['in-sample SR, mu estimated']**2).mean():.3f}")
summary = pd.DataFrame({"mean": res.mean(), "share below 1/N": (res < sharpe(w_eq)).mean()})
summary.loc["max SR (truth)"] = [sr_max, np.nan]
summary.loc["1/N"] = [sharpe(w_eq), np.nan]
summary.round(3)

In [ ]:
pd.DataFrame({"true": w_true, "median plug-in": np.median(W_plug, axis=0),
              "5% quantile": np.quantile(W_plug, 0.05, axis=0), "95% quantile": np.quantile(W_plug, 0.95, axis=0)},
             index=assets).style.format("{:.0%}")

The simulation confirms $E[\widehat{\mathrm{SR}}^2] = 0.298 + 0.8 = 1.098$ exactly. The optimizer reports an average in-sample Sharpe ratio of about 1.06
when both inputs are estimated; the same portfolios have an average true Sharpe ratio of 0.28, and 96% of them do worse than 1/N
(0.48). Estimating $\Sigma$ as well changes little: the damage comes from the means. The weights scatter widely (the 5–95% range
of the CHF bond weight is −712% to +545%, partly because normalizing to a sum of 1 explodes when the raw weights nearly cancel).

### Long-only optimization

Maximize the estimated Sharpe ratio with $w \ge 0$ and $\sum w = 1$, for the first 300 histories.

In [ ]:
def long_only_tangency(m_in, S_in):
    neg_sr = lambda w: -(w @ m_in) / np.sqrt(w @ S_in @ w)
    res_ = minimize(neg_sr, np.full(len(m_in), 1 / len(m_in)), method="SLSQP",
                    bounds=[(0, 1)] * len(m_in), constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}])
    return res_.x

rng_lo = np.random.default_rng(7)
W_lo, sr_lo = [], []
for h in range(300):
    R = mu / 12 + rng_lo.standard_normal((T * 12, N)) @ L.T
    w = long_only_tangency(R.mean(axis=0) * 12 - rf, np.cov(R, rowvar=False) * 12)
    W_lo.append(w)
    sr_lo.append(sharpe(w))
W_lo, sr_lo = np.array(W_lo), np.array(sr_lo)
w_lo_true = long_only_tangency(m, Sigma)
print(f"true long-only optimum: SR {sharpe(w_lo_true):.3f}, holdings > 1%: {np.sum(w_lo_true > 0.01)}")
print(f"estimated long-only: mean true SR {sr_lo.mean():.3f}, below 1/N in {np.mean(sr_lo < sharpe(w_eq)):.0%} of histories")
print(f"assets held (> 1%): mean {np.mean((W_lo > 0.01).sum(axis=1)):.1f}, "
      f"largest weight: mean {W_lo.max(axis=1).mean():.0%}")
pd.Series((W_lo > 0.01).mean(axis=0), index=assets).map(lambda v: f"{v:.0%}").rename("share of histories holding the asset")

The true long-only optimum holds seven assets and reaches a Sharpe ratio of 0.541, almost the unconstrained 0.545. The estimated
long-only portfolios hold 3.1 assets on average, with a largest weight of 66%; their average true Sharpe ratio is 0.459, much
better than the unconstrained plug-in but still below 1/N in 59% of histories. Which assets are held is nearly random: each is
held in roughly 30–60% of histories.

### Re-estimating every year

One 40-year history; each year, estimate on the trailing 10 years and recompute the long-only portfolio.

In [ ]:
R_long = mu / 12 + rng.standard_normal((40 * 12, N)) @ L.T
years_roll = np.arange(10, 41)
W_roll = []
for y in years_roll:
    win = R_long[(y - 10) * 12: y * 12]
    W_roll.append(long_only_tangency(win.mean(axis=0) * 12 - rf, np.cov(win, rowvar=False) * 12))
W_roll = np.array(W_roll)
turnover = 0.5 * np.abs(np.diff(W_roll, axis=0)).sum(axis=1)
print(f"average annual turnover {turnover.mean():.0%}, max {turnover.max():.0%}")

Nothing in the true parameters changes over the 40 years, yet the optimized portfolio turns over 27% a year on average and up
to 87% in one year (figure below).

### How many years would the plug-in need?

Average true Sharpe ratio of the plug-in (estimated $\mu$, true $\Sigma$) against the length of the estimation window.

In [ ]:
T_grid = [5, 10, 20, 40, 80, 160, 320]
sr_by_T = []
for Tg in T_grid:
    E = rng.standard_normal((2_000, N)) @ np.linalg.cholesky(Sigma / Tg).T     # m_hat - m ~ N(0, Sigma / T)
    M_hat = m + E
    W = np.linalg.solve(Sigma, M_hat.T).T
    sr_by_T.append(np.mean((W @ m) / np.sqrt(np.einsum("ij,jk,ik->i", W, Sigma, W))))
pd.Series(sr_by_T, index=pd.Index(T_grid, name="years of data")).round(3).rename("average true SR of plug-in")

With the covariance matrix known, the plug-in reaches the Sharpe ratio of 1/N (0.477) only with about 80 years of data and is
still below the maximum after 320 years. With $\Sigma$ estimated too, and more assets, the required span grows further.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bins = np.linspace(-0.6, 1.8, 61)
ax.hist(res["in-sample SR, both estimated"], bins=bins, histtype="step", linewidth=1.5, label="in-sample SR (what the optimizer reports)")
ax.hist(res["true SR, both estimated"], bins=bins, histtype="step", linewidth=1.5, label="true SR of the same portfolio")
ax.axvline(sr_max, color="black", linewidth=1, label=f"true maximum {sr_max:.2f}")
ax.axvline(sharpe(w_eq), color="tab:green", linestyle="--", label=f"1/N {sharpe(w_eq):.2f}")
ax.set_title("Plug-in tangency portfolios from 10 years of monthly data (1,000 histories)")
ax.set_xlabel("annual Sharpe ratio")
ax.set_ylabel("count")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
pos = np.arange(N)
ax.boxplot([W_plug[:, i] for i in range(N)], positions=pos, widths=0.5, showfliers=False)
ax.plot(pos, w_true, "r*", markersize=10, label="true tangency weights")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(pos)
ax.set_xticklabels(assets, rotation=25, ha="right")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Plug-in tangency weights across 1,000 histories (box: 25–75%, whiskers: 1.5 IQR)")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.stackplot(years_roll, W_roll.T, labels=assets, alpha=0.85)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlim(years_roll[0], years_roll[-1])
ax.set_ylim(0, 1)
ax.set_title("Long-only optimized weights, re-estimated yearly on a trailing 10-year window")
ax.set_xlabel("year of the history (end of the 10-year window)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(T_grid, sr_by_T, "o-", label="plug-in (estimated mu, true Sigma)")
ax.axhline(sr_max, color="black", linewidth=1, label="true maximum")
ax.axhline(sharpe(w_eq), color="tab:green", linestyle="--", label="1/N")
ax.set_xticks(T_grid)
ax.set_xticklabels(T_grid)
ax.set_title("Average true Sharpe ratio of the plug-in portfolio")
ax.set_xlabel("years of data (log scale)")
ax.set_ylabel("annual Sharpe ratio")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
plt.show()

## 7. Exercises

### Basic

1. Redo Example 1 with correlation 0.5. What are the weights?
2. Twenty assets, five years of data, true maximum Sharpe ratio 0.4. What in-sample Sharpe ratio do you expect the optimizer to
   report?
3. An optimizer proposes 45% emerging-market equities, 0% Swiss equities, 130% global bonds and −70% CHF bonds, based on ten years of
   history. **Interpret** this proposal for an investment committee.

### Intermediate

4. Verify $E[\widehat{\mathrm{SR}}^2] = \mathrm{SR}^{*2} + N/T$ by simulation for $T = 5, 10, 20$, with $\Sigma$ known.
5. Compare the true Sharpe ratios of the unconstrained plug-in (both estimated) and the long-only plug-in over the same 300
   histories. Which is better on average, and which has the worse bad cases (5% quantile)?

### Advanced

6. Shrink the estimated excess returns toward their cross-sectional average: $\hat m_\delta = (1-\delta)\hat m + \delta\,\bar{\hat m}\mathbf 1$. Find the $\delta$ that maximizes
   the average true Sharpe ratio (true $\Sigma$, 10 years). How close does it get to the true maximum?
7. Michaud resampling: for one history, draw 200 bootstrap samples of its months, compute the long-only plug-in weights for
   each, and average them. Repeat for 50 histories and compare the average true Sharpe ratio with the long-only plug-in.

## 8. Solutions / Checks

In [ ]:
# 1.
rho = 0.5
x1 = np.linalg.solve(0.01 * np.array([[1, rho], [rho, 1]]), np.array([0.06, 0.04]))
print(f"1. weights {x1 / x1.sum()}")

# 2.
print(f"2. expected in-sample SR ~ sqrt(0.16 + 20/5) = {np.sqrt(0.16 + 4):.2f}")

# 4.
for Tg in [5, 10, 20]:
    E = rng.standard_normal((20_000, N)) @ np.linalg.cholesky(Sigma / Tg).T
    Mh = m + E
    sr2 = np.einsum("ij,ij->i", Mh, np.linalg.solve(Sigma, Mh.T).T)
    print(f"4. T = {Tg:2d}: simulated {sr2.mean():.3f}, theory {sr_max**2 + N / Tg:.3f}")

In [ ]:
# 5.
rng5 = np.random.default_rng(7)                      # same histories as the long-only run
sr_unc = []
for h in range(300):
    R = mu / 12 + rng5.standard_normal((T * 12, N)) @ L.T
    sr_unc.append(sharpe(np.linalg.solve(np.cov(R, rowvar=False) * 12, R.mean(axis=0) * 12 - rf)))
sr_unc = np.array(sr_unc)
print(f"5. unconstrained: mean {sr_unc.mean():.3f}, 5% quantile {np.quantile(sr_unc, 0.05):.3f};  "
      f"long-only: mean {sr_lo.mean():.3f}, 5% quantile {np.quantile(sr_lo, 0.05):.3f}")

# 6.
deltas = np.linspace(0, 1, 21)
E = rng.standard_normal((4_000, N)) @ np.linalg.cholesky(Sigma / 10).T
Mh = m + E
sr_delta = []
for d in deltas:
    Md = (1 - d) * Mh + d * Mh.mean(axis=1, keepdims=True)
    W = np.linalg.solve(Sigma, Md.T).T
    sr_delta.append(np.mean((W @ m) / np.sqrt(np.einsum("ij,jk,ik->i", W, Sigma, W))))
best = int(np.argmax(sr_delta))
print(f"6. best delta {deltas[best]:.2f}: average true SR {sr_delta[best]:.3f} (delta = 0: {sr_delta[0]:.3f}, max {sr_max:.3f})")

In [ ]:
# 7.
rng7 = np.random.default_rng(11)
sr_plain, sr_resampled = [], []
for h in range(50):
    R = mu / 12 + rng7.standard_normal((T * 12, N)) @ L.T
    sr_plain.append(sharpe(long_only_tangency(R.mean(axis=0) * 12 - rf, np.cov(R, rowvar=False) * 12)))
    Wb = []
    for b in range(200):
        Rb = R[rng7.integers(0, R.shape[0], R.shape[0])]
        Wb.append(long_only_tangency(Rb.mean(axis=0) * 12 - rf, np.cov(Rb, rowvar=False) * 12))
    sr_resampled.append(sharpe(np.mean(Wb, axis=0)))
print(f"7. long-only plug-in {np.mean(sr_plain):.3f}, resampled {np.mean(sr_resampled):.3f}, 1/N {sharpe(w_eq):.3f}")

**1.** $w \propto (0.06 - 0.5\cdot 0.04,\ 0.04 - 0.5\cdot 0.06) = (0.04, 0.01)$, i.e. 80% and 20%. Less extreme than with $\rho = 0.9$, still far from 50/50.

**2.** $\sqrt{0.16 + 20/5} = 2.04$: the optimizer would report a Sharpe ratio of about 2 for a portfolio whose true optimum is 0.4.

**3.** The proposal is driven by estimation error: the long-short position between two bond indices with correlation 0.85 is
the signature of a small-eigenvalue spread, and the zero weight in Swiss equities next to a large EM weight reflects ten years
of realized returns, not expectations. The in-sample Sharpe ratio behind it is inflated by about $N/T = 0.8$ in squared terms. The
committee should not implement it; inputs need anchoring (equilibrium returns, 08/04), constraints and a risk-based check.

**4.** The simulated means match $\mathrm{SR}^{*2} + N/T$ for all three horizons.

**5.** The long-only plug-in has the higher average true Sharpe ratio and a much better 5% quantile: the constraint removes
the leveraged spreads that cause the worst outcomes, at the cost of concentration.

**6.** The best $\delta$ is 0.6, which raises the average true Sharpe ratio from 0.281 to 0.368. That is a large gain, but still well
below 1/N (0.477) and the maximum (0.545): shrinking toward a common mean removes much of the noise in the differences between
assets, yet $\Sigma^{-1}$ still turns the remaining noise into spreads. Shrinking toward equilibrium returns (08/04) works better
because the target itself carries information.

**7.** Averaging the resampled long-only portfolios spreads the weight over more assets and raises the average true Sharpe ratio
from 0.458 to 0.491, slightly above 1/N (0.477) in these 50 histories.

## 9. Conclusion

- $\Sigma^{-1}$ divides the estimated return along each eigenvector by its variance; errors along low-variance spreads of similar
  assets become large long-short positions.
- In-sample, the maximum squared Sharpe ratio is inflated by $N/T$; out of sample, the plug-in portfolio falls short of the
  true maximum and often of 1/N.
- With realistic data lengths the plug-in tangency portfolio needs centuries to beat 1/N reliably.
- Long-only constraints remove the leverage but produce concentrated, unstable corner portfolios with high turnover.
- The fixes reduce the noise in the inputs: shrink the covariance matrix (08/02), drop the means (risk parity, 08/03),
  or anchor them to equilibrium (Black–Litterman, 08/04).

Next: [08/02 Covariance shrinkage (Ledoit–Wolf)](./02_covariance_shrinkage_ledoit_wolf.ipynb).